# 🛒 E-commerce Product Recommendation System
## Notebook 3: Train/Test Split + Negative Sampling

**Goal:** Build the final labelled training and test datasets for XGBoost.

### Steps:
1. Load clean data + fix dtypes
2. Time-based train/test split
3. Analyse customer overlap
4. Normalise descriptions + build product list
5. Negative sampling (random, 1:5 ratio)
6. Merge all feature tables
7. Handle NaNs
8. Save final datasets

---
## 1. Imports + Load Data

In [1]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

# Load clean dataset from Notebook 01
df_clean = pd.read_csv('../data/df_clean.csv')

# Fix dtypes lost during CSV save
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['CustomerID']  = df_clean['CustomerID'].astype(str)

# Normalise Description — must be consistent across all notebooks
# Removes punctuation so 'T-LIGHT' becomes 'T LIGHT' everywhere
df_clean['Description'] = df_clean['Description'].apply(
    lambda x: re.sub(r'[^\w\s]', ' ', str(x))
).str.strip().str.upper()

print(f'Loaded: {df_clean.shape[0]:,} rows | {df_clean["CustomerID"].nunique():,} customers | {df_clean["Description"].nunique():,} products')
print(f'Date range: {df_clean["InvoiceDate"].min().date()} → {df_clean["InvoiceDate"].max().date()}')

Loaded: 270,848 rows | 3,936 customers | 3,777 products
Date range: 2010-12-01 → 2011-12-09


---
## 2. Time-Based Train/Test Split

**Why time-based and not random?**  
Random split causes temporal leakage — model learns from future transactions to predict past behaviour.  
We always train on history and evaluate on future — mirroring real deployment.

**Split:**
- Train: Dec 2010 → Oct 2011 (11 months)
- Test: Nov 2011 → Dec 2011 (2 months — covers Christmas gifting peak)

In [2]:
# Hard-coded split date — explicit contract, not fragile date arithmetic
split_date = pd.Timestamp('2011-10-31')

train_df = df_clean[df_clean['InvoiceDate'] <= split_date].copy()
test_df  = df_clean[df_clean['InvoiceDate'] >  split_date].copy()

print(f'Train: {train_df.shape[0]:,} rows | {train_df["InvoiceDate"].min().date()} → {train_df["InvoiceDate"].max().date()}')
print(f'Test:  {test_df.shape[0]:,} rows  | {test_df["InvoiceDate"].min().date()} → {test_df["InvoiceDate"].max().date()}')
print(f'Train customers: {train_df["CustomerID"].nunique():,}')
print(f'Test customers:  {test_df["CustomerID"].nunique():,}')

Train: 214,913 rows | 2010-12-01 → 2011-10-30
Test:  55,935 rows  | 2011-10-31 → 2011-12-09
Train customers: 3,603
Test customers:  1,718


---
## 3. Customer Overlap Analysis

Customers who appear **only in test** (never seen in training) are cold start users.  
Our model has zero feature data for them — no RFM, no category affinity, no purchase history.  
→ These customers receive **popularity-based fallback** recommendations, not XGBoost.

We evaluate XGBoost **only on customers seen in both train and test.**

In [3]:
train_customers = set(train_df['CustomerID'].unique())
test_customers  = set(test_df['CustomerID'].unique())

overlap      = train_customers & test_customers   # in both
only_in_test = test_customers - train_customers   # cold start

print(f'Customers in both train and test: {len(overlap):,}')
print(f'Customers only in test (cold start → popularity fallback): {len(only_in_test):,}')
print(f'Overlap %: {len(overlap)/len(test_customers)*100:.1f}%')

# Filter test to overlap customers only — can only evaluate where we have features
test_df = test_df[test_df['CustomerID'].isin(overlap)].copy()
print(f'\nFinal test_df after filtering: {test_df.shape[0]:,} rows | {test_df["CustomerID"].nunique():,} customers')

Customers in both train and test: 1,385
Customers only in test (cold start → popularity fallback): 333
Overlap %: 80.6%

Final test_df after filtering: 42,967 rows | 1,385 customers


---
## 4. Load Products + Normalise Descriptions

⚠️ **Critical step — must happen BEFORE negative sampling.**

Product descriptions must be normalised consistently everywhere:
- `df_clean` had punctuation removed in Notebook 02
- `products.csv` may still have original punctuation
- Mismatch causes failed joins → NaN product features

Fix: normalise `products.csv` descriptions before building `all_products_set`.

In [4]:
# Load and normalise products — same cleaning as df_clean
products = pd.read_csv('../data/products.csv')
products['Description'] = products['Description'].apply(
    lambda x: re.sub(r'[^\w\s]', ' ', str(x))
).str.strip().str.upper()

# Build product set for negative sampling
all_products_set = set(products['Description'].unique())

print(f'Total products available for negative sampling: {len(all_products_set):,}')

Total products available for negative sampling: 3,777


---
## 5. Negative Sampling

**The problem:** Transaction data only contains positive examples (products customers DID buy).  
We have no explicit 'did not buy' signal — we must construct negatives artificially.

**Why not use all unobserved pairs as negatives?**  
97.6% of the user-item matrix is zero — most zeros are unobserved, not true negatives.  
Training on all zeros produces a model that always predicts 0.

**Random negative sampling (1:5 ratio):**
- For each customer: sample 5× their bought products from never-bought products
- Label positives = 1, negatives = 0
- 1:5 ratio is industry standard — balances learning both class boundaries

In [5]:
# ── Negative Sampling — Training Data ─────────────────────────────────
np.random.seed(42)  # reproducibility
training_rows = []

for customer_id, group in train_df.groupby('CustomerID'):
    # Products this customer HAS bought (positive class)
    bought_products = set(group['Description'].unique())
    
    # Products this customer has NEVER bought
    never_bought = list(all_products_set - bought_products)
    
    # Sample negatives at 1:5 ratio
    # min() handles edge case where customer bought almost all products
    num_negatives = min(len(never_bought), len(bought_products) * 5)
    negative_samples = np.random.choice(never_bought, num_negatives, replace=False)
    
    # Add negatives → label 0
    for prod in negative_samples:
        training_rows.append({'CustomerID': customer_id, 'Description': prod, 'label': 0})
    
    # Add positives → label 1
    for prod in bought_products:
        training_rows.append({'CustomerID': customer_id, 'Description': prod, 'label': 1})

train_data = pd.DataFrame(training_rows)

print(f'Train data shape: {train_data.shape}')
print(f'Label distribution:')
print(train_data['label'].value_counts())
print(f'Ratio: 1:{train_data["label"].value_counts()[0] // train_data["label"].value_counts()[1]}')

Train data shape: (1025232, 3)
Label distribution:
label
0    854360
1    170872
Name: count, dtype: int64
Ratio: 1:5


In [6]:
# ── Negative Sampling — Test Data ─────────────────────────────────────
# Same logic — samples negatives from test period purchases
# Only for overlap customers (those seen in training)
test_rows = []

for customer_id, group in test_df.groupby('CustomerID'):
    bought_products = set(group['Description'].unique())
    never_bought    = list(all_products_set - bought_products)
    
    num_negatives    = min(len(never_bought), len(bought_products) * 5)
    negative_samples = np.random.choice(never_bought, num_negatives, replace=False)
    
    for prod in negative_samples:
        test_rows.append({'CustomerID': customer_id, 'Description': prod, 'label': 0})
    
    for prod in bought_products:
        test_rows.append({'CustomerID': customer_id, 'Description': prod, 'label': 1})

test_data = pd.DataFrame(test_rows)

print(f'Test data shape: {test_data.shape}')
print(f'Label distribution:')
print(test_data['label'].value_counts())
print(f'Ratio: 1:{test_data["label"].value_counts()[0] // test_data["label"].value_counts()[1]}')

Test data shape: (230496, 3)
Label distribution:
label
0    192080
1     38416
Name: count, dtype: int64
Ratio: 1:5


---
## 6. Merge Feature Tables

We merge 4 feature tables onto train_data and test_data.

| Table | Join Key | Adds |
|---|---|---|
| `rfm` | CustomerID | Recency, Frequency, Monetary |
| `products` | Description | TotalUnitsSold, UniqueBuyers, AvgTransactionValue, Category |
| `customer_category` | CustomerID + Category | CategoryAffinity, CategoryCount |
| `customer_products` | CustomerID + Description | TimesBought |

**Why left join?**  
We keep all (customer, product) pairs we constructed — including negative samples that may not exist in feature tables.

In [7]:
# Load all feature tables
rfm               = pd.read_csv('../data/rfm.csv')
customer_category = pd.read_csv('../data/customer_category.csv')
customer_products = pd.read_csv('../data/customer_products.csv')

# Normalise products descriptions (already loaded and normalised above)
# Fix dtypes
for df in [rfm, customer_category, customer_products]:
    df['CustomerID'] = df['CustomerID'].astype(str)

# Normalise customer_products Description too
customer_products['Description'] = customer_products['Description'].apply(
    lambda x: re.sub(r'[^\w\s]', ' ', str(x))
).str.strip().str.upper()

print('Feature tables loaded:')
print(f'  rfm:               {rfm.shape}')
print(f'  products:          {products.shape}')
print(f'  customer_category: {customer_category.shape}')
print(f'  customer_products: {customer_products.shape}')

Feature tables loaded:
  rfm:               (3936, 5)
  products:          (3779, 5)
  customer_category: (39619, 5)
  customer_products: (209114, 4)


In [8]:
# ── Merge features onto train_data ────────────────────────────────────
train_data['CustomerID'] = train_data['CustomerID'].astype(str)

# 1. RFM — user-level features (join on CustomerID)
train_data = train_data.merge(rfm, on='CustomerID', how='left')

# 2. Products — product-level features (join on Description)
#    Also adds Category column — needed for next merge
train_data = train_data.merge(products, on='Description', how='left')

# 3. Category affinity — join on CustomerID + Category
#    NaN when customer never bought from this product's category
train_data = train_data.merge(
    customer_category[['CustomerID', 'Category', 'CategoryAffinity', 'CategoryCount']],
    on=['CustomerID', 'Category'], how='left'
)

# 4. Purchase history — join on CustomerID + Description
#    NaN for negative samples (never bought this product)
train_data = train_data.merge(
    customer_products[['CustomerID', 'Description', 'TimesBought']],
    on=['CustomerID', 'Description'], how='left'
)

print(f'Train data shape after merges: {train_data.shape}')
print(f'Columns: {train_data.columns.tolist()}')

Train data shape after merges: (1025768, 14)
Columns: ['CustomerID', 'Description', 'label', 'LastPurchaseDate', 'Frequency', 'Monetary', 'Recency', 'TotalUnitsSold', 'UniqueBuyers', 'AvgTransactionValue', 'Category', 'CategoryAffinity', 'CategoryCount', 'TimesBought']


In [9]:
# ── Merge features onto test_data ─────────────────────────────────────
test_data['CustomerID'] = test_data['CustomerID'].astype(str)

test_data = test_data.merge(rfm, on='CustomerID', how='left')
test_data = test_data.merge(products, on='Description', how='left')
test_data = test_data.merge(
    customer_category[['CustomerID', 'Category', 'CategoryAffinity', 'CategoryCount']],
    on=['CustomerID', 'Category'], how='left'
)
test_data = test_data.merge(
    customer_products[['CustomerID', 'Description', 'TimesBought']],
    on=['CustomerID', 'Description'], how='left'
)

print(f'Test data shape after merges: {test_data.shape}')
print(f'Columns: {test_data.columns.tolist()}')

Test data shape after merges: (230601, 14)
Columns: ['CustomerID', 'Description', 'label', 'LastPurchaseDate', 'Frequency', 'Monetary', 'Recency', 'TotalUnitsSold', 'UniqueBuyers', 'AvgTransactionValue', 'Category', 'CategoryAffinity', 'CategoryCount', 'TimesBought']


---
## 7. Handle NaN Values

NaNs appear for legitimate reasons — not data errors:

| Column | Why NaN | Fill Value |
|---|---|---|
| `TimesBought` | Negative samples — customer never bought this product | 0 |
| `CategoryAffinity` | Customer never bought from this product's category | 0 |
| `CategoryCount` | Same reason as CategoryAffinity | 0 |
| `Category` | Product description didn't match any keyword | 'OTHER' |
| `TotalUnitsSold` etc. | Product not in products.csv (rare edge case) | median |

In [10]:
# ── Fill NaNs with meaningful defaults ────────────────────────────────
for df in [train_data, test_data]:
    # 0 = never bought / no affinity for this category
    df['TimesBought']      = df['TimesBought'].fillna(0)
    df['CategoryAffinity'] = df['CategoryAffinity'].fillna(0)
    df['CategoryCount']    = df['CategoryCount'].fillna(0)
    
    # OTHER = no category match
    df['Category'] = df['Category'].fillna('OTHER')
    
    # Product features — fill with median (product exists but not in products.csv)
    for col in ['TotalUnitsSold', 'UniqueBuyers', 'AvgTransactionValue']:
        df[col] = df[col].fillna(df[col].median())

print('NaNs remaining in train_data:')
remaining = train_data.isnull().sum()
print(remaining[remaining > 0] if remaining[remaining > 0].shape[0] > 0 else '  None ✅')

print('\nNaNs remaining in test_data:')
remaining = test_data.isnull().sum()
print(remaining[remaining > 0] if remaining[remaining > 0].shape[0] > 0 else '  None ✅')

NaNs remaining in train_data:
  None ✅

NaNs remaining in test_data:
  None ✅


---
## 8. Final Check + Save

Before saving — verify the final datasets are clean and ready for XGBoost.

In [11]:
# ── Final verification ─────────────────────────────────────────────────
print('=' * 55)
print('     FINAL DATASET SUMMARY')
print('=' * 55)
print(f'Train shape:       {train_data.shape}')
print(f'Test shape:        {test_data.shape}')
print(f'Features:          {train_data.shape[1] - 3} (excl. CustomerID, Description, label)')
print('-' * 55)
print(f'Train positives:   {train_data["label"].sum():,}')
print(f'Train negatives:   {(train_data["label"]==0).sum():,}')
print(f'Train ratio:       1:{(train_data["label"]==0).sum()//train_data["label"].sum()}')
print('-' * 55)
print(f'Test positives:    {test_data["label"].sum():,}')
print(f'Test negatives:    {(test_data["label"]==0).sum():,}')
print('-' * 55)
print('Feature columns:')
feature_cols = [c for c in train_data.columns if c not in ['CustomerID','Description','label','LastPurchaseDate']]
for c in feature_cols:
    print(f'  {c}')
print('=' * 55)

train_data.head()

     FINAL DATASET SUMMARY
Train shape:       (1025768, 14)
Test shape:        (230601, 14)
Features:          11 (excl. CustomerID, Description, label)
-------------------------------------------------------
Train positives:   170,915
Train negatives:   854,853
Train ratio:       1:5
-------------------------------------------------------
Test positives:    38,422
Test negatives:    192,179
-------------------------------------------------------
Feature columns:
  Frequency
  Monetary
  Recency
  TotalUnitsSold
  UniqueBuyers
  AvgTransactionValue
  Category
  CategoryAffinity
  CategoryCount
  TimesBought


,CustomerID,Description,label,LastPurchaseDate,Frequency,Monetary,Recency,TotalUnitsSold,UniqueBuyers,AvgTransactionValue,Category,CategoryAffinity,CategoryCount,TimesBought
0,12347,HANGING WOOD AND FELT FLOWER,0,2011-12-07 15:52:00,182,4310.0,2,1099,26,5.138571,HOME_DECOR,0.170330,31.0,0.0
1,12347,ASS COL LARGE SAND FROG P WEIGHT,0,2011-12-07 15:52:00,182,4310.0,2,27,6,4.050000,OTHER,0.000000,0.0,0.0
2,12347,WOOD S 3 CABINET ANT WHITE FINISH,0,2011-12-07 15:52:00,182,4310.0,2,918,145,30.459839,OTHER,0.000000,0.0,0.0
3,12347,BLOND DOLL DOORSTOP,0,2011-12-07 15:52:00,182,4310.0,2,91,10,7.950000,TOYS,0.000000,0.0,0.0
4,12347,LILAC GAUZE BUTTERFLY LAMPSHADE,0,2011-12-07 15:52:00,182,4310.0,2,9,2,1.890000,OTHER,0.192308,35.0,0.0


In [12]:
# ── Save final datasets ────────────────────────────────────────────────
train_data.to_csv('../data/train_data.csv', index=False)
test_data.to_csv('../data/test_data.csv', index=False)

print('✅ Saved:')
print('   data/train_data.csv  → XGBoost training dataset')
print('   data/test_data.csv   → XGBoost evaluation dataset')
print()
print('Next → Notebook 04: XGBoost Training + Optuna + SHAP')

✅ Saved:
   data/train_data.csv  → XGBoost training dataset
   data/test_data.csv   → XGBoost evaluation dataset

Next → Notebook 04: XGBoost Training + Optuna + SHAP
